## weights-genre-tags.ipynb
Builds a unified genre tag matrix by combining three tag sources:
- **Album tags** (direct, weight 1.0) — from `mb_album_tag.parquet`
- **Artist tags** (weight 0.5, only for albums with < 5 direct tags) — from `mb_artist_tag.parquet` joined via `mb_album_artists.parquet`
- **Label tags** (weight 0.3) — from `mb_album_label.parquet`

Tags with fewer than 10 occurrences across all sources are dropped.
Per-row L1 normalisation is applied after combining so each album's tag profile sums to 1.0.
Output saved to `data/features/album_genre_matrix.npz`, replacing the existing `album_tags_matrix.npz` in the feature stack.

In [1]:
import pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.preprocessing import normalize

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'

ALBUM_TAG_MIN   = 5    # albums with fewer direct tags than this get artist tags blended in
MIN_TAG_OCC     = 10   # tags appearing fewer times than this across all sources are dropped
W_ALBUM         = 1.0
W_ARTIST        = 0.5
W_LABEL         = 0.3

In [2]:
# Load master album index
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)
album_index = pd.Index(album_ids)
n_albums = len(album_index)

with open(f'{FEATURES_DIR}/artist_ids.pkl', 'rb') as f:
    artist_ids = pickle.load(f)
artist_index = pd.Index(artist_ids)

print(f'Album universe : {n_albums:,}')
print(f'Artist universe: {len(artist_index):,}')

Album universe : 1,008,102
Artist universe: 245,407


In [3]:
# --- Album tags ---
album_tags = pd.read_parquet(f'{DATA_DIR}/mb_album_tag.parquet')
album_tags = album_tags[album_tags['tag_count'] > 0].copy()
print(f'Album tag rows (positive votes): {len(album_tags):,}')

# --- Artist tags ---
artist_tags = pd.read_parquet(f'{DATA_DIR}/mb_artist_tag.parquet')
artist_tags = artist_tags[artist_tags['tag_count'] > 0].copy()
print(f'Artist tag rows (positive votes): {len(artist_tags):,}')

# --- Label tags (embedded in album_label parquet) ---
label_tags = pd.read_parquet(f'{DATA_DIR}/mb_album_label.parquet',
                              columns=['album_id', 'tag_id', 'tag_count'])
label_tags = label_tags[label_tags['tag_count'] > 0].copy()
print(f'Label tag rows (positive votes): {len(label_tags):,}')

Album tag rows (positive votes): 3,042,637
Artist tag rows (positive votes): 702,381
Label tag rows (positive votes): 737,120


In [4]:
# Build a unified tag vocabulary — only tags that appear >= MIN_TAG_OCC times
# across all three sources combined
album_artists = (
    pd.read_parquet(f'{DATA_DIR}/mb_album_artists.parquet', columns=['album_id', 'artist_id'])
    .drop_duplicates(subset='album_id')
)

# Map artist tags to albums so we can count occurrences at album level
artist_tags_on_albums = (
    album_artists
    .merge(artist_tags, on='artist_id', how='inner')
    [['album_id', 'tag_id', 'tag_count']]
)

all_tags = pd.concat([
    album_tags[['tag_id', 'tag_count']],
    artist_tags_on_albums[['tag_id', 'tag_count']],
    label_tags[['tag_id', 'tag_count']],
], ignore_index=True)

tag_occ = all_tags.groupby('tag_id')['tag_count'].sum()
popular_tags = tag_occ[tag_occ >= MIN_TAG_OCC].index
tag_index = pd.Index(sorted(popular_tags))
n_tags = len(tag_index)

print(f'Total unique tags across all sources : {tag_occ.shape[0]:,}')
print(f'Tags kept (>= {MIN_TAG_OCC} occurrences)          : {n_tags:,}')

Total unique tags across all sources : 50,305
Tags kept (>= 10 occurrences)          : 11,247


In [5]:
# Identify albums with fewer than ALBUM_TAG_MIN direct tags — these get artist tags blended in
X_existing = load_npz(f'{FEATURES_DIR}/album_tags_matrix.npz')
tags_per_album = np.diff(X_existing.indptr)
sparse_album_mask = tags_per_album < ALBUM_TAG_MIN
sparse_album_ids = album_index[sparse_album_mask]

print(f'Albums with < {ALBUM_TAG_MIN} direct tags (will get artist blend): {sparse_album_mask.sum():,} ({sparse_album_mask.mean()*100:.1f}%)')
print(f'Albums with >= {ALBUM_TAG_MIN} direct tags (album tags only)     : {(~sparse_album_mask).sum():,} ({(~sparse_album_mask).mean()*100:.1f}%)')

Albums with < 5 direct tags (will get artist blend): 833,402 (82.7%)
Albums with >= 5 direct tags (album tags only)     : 174,700 (17.3%)


In [6]:
# --- BLOCK 1: Album tags (all albums, weight W_ALBUM) ---
at = album_tags[album_tags['tag_id'].isin(popular_tags)].copy()

# Normalise tag weights per album
at_totals = at.groupby('album_id')['tag_count'].transform('sum')
at['weight'] = (at['tag_count'] / at_totals * W_ALBUM).astype('float32')

row_idx = album_index.get_indexer(at['album_id'].values)
col_idx = tag_index.get_indexer(at['tag_id'].values)
valid = (row_idx >= 0) & (col_idx >= 0)

X_album = csr_matrix(
    (at['weight'].values[valid], (row_idx[valid], col_idx[valid])),
    shape=(n_albums, n_tags)
)
print(f'Album block  : {X_album.shape}  nnz={X_album.nnz:,}')

Album block  : (1008102, 11247)  nnz=3,015,393


In [7]:
# --- BLOCK 2: Artist tags (sparse albums only, weight W_ARTIST) ---
art = (
    album_artists[album_artists['album_id'].isin(sparse_album_ids)]
    .merge(artist_tags[artist_tags['tag_id'].isin(popular_tags)], on='artist_id', how='inner')
    [['album_id', 'tag_id', 'tag_count']]
)

art_totals = art.groupby('album_id')['tag_count'].transform('sum')
art['weight'] = (art['tag_count'] / art_totals * W_ARTIST).astype('float32')

row_idx = album_index.get_indexer(art['album_id'].values)
col_idx = tag_index.get_indexer(art['tag_id'].values)
valid = (row_idx >= 0) & (col_idx >= 0)

X_artist = csr_matrix(
    (art['weight'].values[valid], (row_idx[valid], col_idx[valid])),
    shape=(n_albums, n_tags)
)
print(f'Artist block : {X_artist.shape}  nnz={X_artist.nnz:,}')

Artist block : (1008102, 11247)  nnz=2,189,734


In [8]:
# --- BLOCK 3: Label tags (all albums, weight W_LABEL) ---
lt = label_tags[label_tags['tag_id'].isin(popular_tags)].copy()

lt_totals = lt.groupby('album_id')['tag_count'].transform('sum')
lt['weight'] = (lt['tag_count'] / lt_totals * W_LABEL).astype('float32')

row_idx = album_index.get_indexer(lt['album_id'].values)
col_idx = tag_index.get_indexer(lt['tag_id'].values)
valid = (row_idx >= 0) & (col_idx >= 0)

X_label = csr_matrix(
    (lt['weight'].values[valid], (row_idx[valid], col_idx[valid])),
    shape=(n_albums, n_tags)
)
print(f'Label block  : {X_label.shape}  nnz={X_label.nnz:,}')

Label block  : (1008102, 11247)  nnz=400,902


In [9]:
# --- Combine and L1-normalise ---
# Summing sparse matrices accumulates weights per (album, tag) cell across all sources
X_genre = X_album + X_artist + X_label

# L1 normalise so each album's tag profile sums to 1.0
X_genre = normalize(X_genre, norm='l1', axis=1)

print(f'Genre matrix : {X_genre.shape}  nnz={X_genre.nnz:,}')
print(f'Albums with any genre signal: {(np.diff(X_genre.indptr) > 0).sum():,}')

# Sanity: compare coverage to original album tags alone
original_covered = (np.diff(X_existing.indptr) > 0).sum()
genre_covered    = (np.diff(X_genre.indptr) > 0).sum()
print(f'Coverage vs original album_tags_matrix: {original_covered:,} → {genre_covered:,}')

Genre matrix : (1008102, 11247)  nnz=5,015,180
Albums with any genre signal: 1,008,102
Coverage vs original album_tags_matrix: 1,008,102 → 1,008,102


In [10]:
# Spot-check: compare tag profiles for a sparse album before and after
sparse_example_idx = np.where(sparse_album_mask)[0][0]
sparse_example_id  = album_index[sparse_example_idx]

old_nnz = X_existing[sparse_example_idx].nnz
new_nnz = X_genre[sparse_example_idx].nnz
print(f'Example sparse album id={sparse_example_id}: {old_nnz} tags before → {new_nnz} tags after')

# Distribution of tag counts after enrichment for previously-sparse albums
new_tags_per_sparse = np.diff(X_genre.indptr)[sparse_album_mask]
print(f'\nTag count distribution for previously-sparse albums (after enrichment):')
for n in [0, 1, 2, 3, 4, 5, 10]:
    c = (new_tags_per_sparse <= n).sum()
    print(f'  <= {n:>2} tags: {c:>10,}  ({c/len(new_tags_per_sparse)*100:.1f}%)')

Example sparse album id=11: 4 tags before → 6 tags after

Tag count distribution for previously-sparse albums (after enrichment):
  <=  0 tags:          0  (0.0%)
  <=  1 tags:     94,520  (11.3%)
  <=  2 tags:    251,695  (30.2%)
  <=  3 tags:    413,857  (49.7%)
  <=  4 tags:    555,057  (66.6%)
  <=  5 tags:    633,009  (76.0%)
  <= 10 tags:    773,377  (92.8%)


In [11]:
save_npz(f'{FEATURES_DIR}/album_genre_matrix.npz', X_genre)
print(f'Saved: {FEATURES_DIR}/album_genre_matrix.npz')
print(f'Shape : {X_genre.shape}')
print(f'nnz   : {X_genre.nnz:,}')

Saved: ../data/features/album_genre_matrix.npz
Shape : (1008102, 11247)
nnz   : 5,015,180
